# EfficientNet-B0 con manifiestos (baseline + fine-tuning) — PlantVillage

Réplica en Colab del pipeline de `src/leafthealthvision` (CRISP-ML(Q), metodología por manifiestos):

1. **Datos**: dataset oficial + partición por manifiestos `data/splits/*.csv` (agrupada por hoja, sin fuga).
2. **Baseline**: EfficientNetB0 ImageNet congelada + cabeza de clasificación.
3. **Fine-tuning**: descongelar las últimas N capas (BatchNorm congelada) y ajustar con lr bajo.
4. **Evaluación**: accuracy, P/R/F1 macro y weighted, matriz de confusión (mismas métricas que la Persona A).
5. **Errores**: CSV de mal clasificados, galería de ejemplos y pares de confusión más frecuentes.

Preprocesamiento: EfficientNet de Keras normaliza internamente (rescale + normalización
dentro del modelo); las imágenes se entregan como float32 en `[0, 255]`, sin `1/255` externo
(a diferencia de MobileNetV2, que sí lo necesita).

> **Importante**: sube `train.csv`, `val.csv` y `test.csv` desde `data/splits/` del repositorio cuando se
> pidan. Si no están, el notebook genera un split estratificado automático **sin agrupación por hoja**;
> esas métricas no son comparables 1:1 con el pipeline local y queda registrado como limitación.

**Runtime recomendado:** GPU (T4).

In [ ]:
# 0. Dependencias
!pip -q install -U tensorflow matplotlib scikit-learn mlflow codecarbon

In [ ]:
# 1. Imports, reproducibilidad y configuración editable
import csv
import random
import shutil
import subprocess
import time
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import applications, layers

SEED = 42
BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)
EPOCHS_BASELINE = 10        # el protocolo local usa 50 (configs/efficientnetb0_baseline.yaml)
EPOCHS_FINETUNE = 5
LEARNING_RATE = 1e-3
FT_LEARNING_RATE = 1e-5
FT_UNFREEZE_LAST_N = 20
EARLY_STOPPING_PATIENCE = 5
FT_EARLY_STOPPING_PATIENCE = 3

WORK = Path('/content/leafthealthvision')
DATA_DIR = WORK / 'data' / 'plantvillage'
COLOR_DIR = DATA_DIR / 'raw' / 'color'
MANIFEST_DIR = WORK / 'data' / 'splits'
ARTIFACTS = WORK / 'artifacts'
ARTIFACTS.mkdir(parents=True, exist_ok=True)
REPO_URL = 'https://github.com/spMohanty/PlantVillage-Dataset.git'

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Control de memoria en GPU: crecimiento gradual + precisión mixta.
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print(f'GPU: {len(gpus)} dispositivo(s), mixed_float16')
else:
    print('Sin GPU; se usará CPU')

print({'tensorflow': tf.__version__, 'image_size': IMAGE_SIZE, 'batch_size': BATCH_SIZE})

In [ ]:
# 2. Descargar PlantVillage (solo la primera vez)
if not COLOR_DIR.exists():
    WORK.mkdir(parents=True, exist_ok=True)
    print(f'Clonando {REPO_URL} ...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(DATA_DIR)], check=True)
    git_dir = DATA_DIR / '.git'
    if git_dir.exists():
        shutil.rmtree(git_dir)
    keep = {'raw', 'leaf_grouping', 'leaf-map.json', 'README.md', 'CITATION.cff'}
    for item in list(DATA_DIR.iterdir()):
        if item.name not in keep:
            shutil.rmtree(item) if item.is_dir() else item.unlink()
else:
    print(f'Dataset ya descargado en: {DATA_DIR}')

if not COLOR_DIR.exists():
    raise FileNotFoundError(f'No existe el directorio esperado: {COLOR_DIR}')

print('Clases en color:', len([p for p in COLOR_DIR.iterdir() if p.is_dir()]))

In [ ]:
# 3. Manifiestos de partición (misma metodología que el repo)
# Los manifiestos referencia nombres de la variante `segmented`; se resuelven a `raw/color`
# por (categoría, UUID + nombre). El repo aplica además un fallback por nombre núcleo
# cuando el UUID no coincide; aquí se replica. Filas sin equivalente quedan registradas.
SUFFIX = '_final_masked'
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp'}


def normalized_name(filename: str) -> str:
    return Path(filename).stem.lower().removesuffix(SUFFIX)


def core_name(filename: str) -> str:
    return normalized_name(filename).split('___', 1)[-1]


MANIFESTS_FROM_REPO = all((MANIFEST_DIR / f'{name}.csv').exists() for name in ('train', 'val', 'test'))

if not MANIFESTS_FROM_REPO:
    print('No se encontraron los manifiestos del repo. Sube train.csv, val.csv y test.csv (data/splits/).')
    try:
        from google.colab import files

        uploaded = files.upload()
        MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
        for name in list(uploaded):
            shutil.move(name, MANIFEST_DIR / name)
        MANIFESTS_FROM_REPO = all((MANIFEST_DIR / f'{n}.csv').exists() for n in ('train', 'val', 'test'))
    except Exception as exc:  # noqa: BLE001
        print(f'Sube manual no disponible ({exc}).')

if not MANIFESTS_FROM_REPO:
    # Limitación: split estratificado por clase SIN agrupación por hoja (riesgo de fuga por hoja).
    print('Generando split estratificado 80/10/10 sin agrupación por hoja (limitación documentada).')
    rows = [
        {'archivo': p.name, 'categoria': p.parent.name}
        for p in sorted(COLOR_DIR.glob('*/*'))
        if p.suffix.lower() in IMAGE_EXTENSIONS
    ]
    df = pd.DataFrame(rows)
    rng = np.random.default_rng(SEED)
    parts = []
    for _, group in df.groupby('categoria', sort=True):
        group = group.iloc[rng.permutation(len(group))].copy()
        n = len(group)
        n_tr, n_va = int(n * 0.8), int(n * 0.1)
        group['split'] = 'test'
        group.iloc[:n_tr, group.columns.get_loc('split')] = 'train'
        group.iloc[n_tr:n_tr + n_va, group.columns.get_loc('split')] = 'val'
        parts.append(group)
    out = pd.concat(parts)
    MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
    for name in ('train', 'val', 'test'):
        out[out['split'] == name][['archivo', 'categoria']].to_csv(MANIFEST_DIR / f'{name}.csv', index=False)


def load_manifest(name: str) -> list[dict]:
    with (MANIFEST_DIR / f'{name}.csv').open(newline='', encoding='utf-8') as handle:
        return list(csv.DictReader(handle))


index, core_index = {}, defaultdict(list)
for class_dir in sorted(p for p in COLOR_DIR.iterdir() if p.is_dir()):
    for img in class_dir.iterdir():
        if img.is_file() and img.suffix.lower() in IMAGE_EXTENSIONS:
            index[(class_dir.name, normalized_name(img.name))] = img
            core_index[(class_dir.name, core_name(img.name))].append(img)

class_names = sorted({row['categoria'] for name in ('train', 'val', 'test') for row in load_manifest(name)})
class_to_id = {name: i for i, name in enumerate(class_names)}

splits, unresolved = {}, []
for name in ('train', 'val', 'test'):
    paths, labels = [], []
    for row in load_manifest(name):
        path = index.get((row['categoria'], normalized_name(row['archivo'])))
        if path is None:
            candidates = core_index.get((row['categoria'], core_name(row['archivo'])), [])
            if len(candidates) == 1:
                path = candidates[0]
        if path is None:
            unresolved.append((name, f"{row['categoria']}/{row['archivo']}"))
            continue
        paths.append(str(path))
        labels.append(class_to_id[row['categoria']])
    splits[name] = {'paths': paths, 'labels': labels}

print({name: len(s['paths']) for name, s in splits.items()}, '| clases:', len(class_names))
if unresolved:
    print(f'Filas sin equivalente en color: {len(unresolved)} (registradas, nunca sustituidas)')
    with (ARTIFACTS / 'unresolved_color_manifest_rows.csv').open('w', newline='', encoding='utf-8') as handle:
        writer = csv.writer(handle)
        writer.writerow(['split', 'manifest_image'])
        writer.writerows(unresolved)

In [ ]:
# 4. tf.data: decode -> float32 [0, 255] (EfficientNet normaliza adentro) -> resize -> batch
AUTOTUNE = tf.data.AUTOTUNE


def make_ds(split: str, training: bool = False):
    ds = tf.data.Dataset.from_tensor_slices((splits[split]['paths'], splits[split]['labels']))
    if training:
        ds = ds.shuffle(min(len(splits[split]['paths']), 10_000), seed=SEED, reshuffle_each_iteration=True)

    def _load(path, label):
        img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
        img.set_shape([None, None, 3])
        img = tf.image.resize(tf.cast(img, tf.float32), IMAGE_SIZE, antialias=True)
        return img, label

    return ds.map(_load, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)


train_ds, val_ds, test_ds = make_ds('train', True), make_ds('val'), make_ds('test')

In [ ]:
# 5. Pesos por clase calculados solo con train: N / (K * n_clase)
counts = Counter(splits['train']['labels'])
class_weight = {c: len(splits['train']['labels']) / (len(class_names) * n) for c, n in counts.items()}

In [ ]:
# 6. MLflow local + registro de métricas por época
import mlflow
from codecarbon import EmissionsTracker

mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('leafthealthvision-efficientnetb0')


class MlflowEpochLogger(tf.keras.callbacks.Callback):
    """Registra métricas de cada época en MLflow."""

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        mlflow.log_metrics({k: float(v) for k, v in logs.items()}, step=epoch)

In [ ]:
# 7. Evaluación compartida: métricas de Persona A + matriz de confusión + errores
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support


def evaluate_stage(model, tag: str):
    probabilities = model.predict(test_ds, verbose=1)
    y_pred = probabilities.argmax(axis=1)
    y_true = np.array(splits['test']['labels'])
    macro = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    weighted = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
    metrics = {
        'accuracy': float(np.mean(y_true == y_pred)),
        'precision_macro': float(macro[0]),
        'recall_macro': float(macro[1]),
        'f1_macro': float(macro[2]),
        'precision_weighted': float(weighted[0]),
        'recall_weighted': float(weighted[1]),
        'f1_weighted': float(weighted[2]),
    }

    matrix = confusion_matrix(y_true, y_pred, labels=np.arange(len(class_names)))
    figure, axis = plt.subplots(figsize=(14, 12))
    image = axis.imshow(matrix, interpolation='nearest', cmap='Blues')
    figure.colorbar(image, ax=axis)
    axis.set(xticks=np.arange(len(class_names)), yticks=np.arange(len(class_names)),
             xlabel='Predicción', ylabel='Clase real', title=f'Matriz de confusión — {tag}')
    axis.set_xticklabels(class_names, rotation=90, fontsize=6)
    axis.set_yticklabels(class_names, fontsize=6)
    figure.tight_layout()
    matrix_path = ARTIFACTS / f'confusion_matrix_{tag}.png'
    figure.savefig(matrix_path, dpi=150)
    plt.show()
    plt.close(figure)

    report = classification_report(y_true, y_pred, target_names=class_names, zero_division=0)
    (ARTIFACTS / f'classification_report_{tag}.txt').write_text(report, encoding='utf-8')

    incorrect = np.flatnonzero(y_pred != y_true)
    with (ARTIFACTS / f'misclassifications_{tag}.csv').open('w', newline='', encoding='utf-8') as handle:
        writer = csv.writer(handle)
        writer.writerow(['path', 'true_class', 'predicted_class', 'confidence'])
        for i in incorrect:
            writer.writerow([splits['test']['paths'][i], class_names[y_true[i]],
                             class_names[y_pred[i]], float(probabilities[i, y_pred[i]])])
    if len(incorrect):
        selected = incorrect[:36]
        cols = 6
        rows_n = int(np.ceil(len(selected) / cols))
        figure, axes = plt.subplots(rows_n, cols, figsize=(18, 3 * rows_n), squeeze=False)
        for ax, i in zip(axes.flat, selected):
            ax.imshow(Image.open(splits['test']['paths'][i]).convert('RGB'))
            ax.set_title(f'real: {class_names[y_true[i]]}\npred: {class_names[y_pred[i]]}', fontsize=7)
            ax.axis('off')
        for ax in axes.flat[len(selected):]:
            ax.axis('off')
        figure.tight_layout()
        figure.savefig(ARTIFACTS / f'misclassifications_{tag}.png', dpi=150)
        plt.show()
        plt.close(figure)

    for path in (matrix_path,
                 ARTIFACTS / f'classification_report_{tag}.txt',
                 ARTIFACTS / f'misclassifications_{tag}.csv',
                 ARTIFACTS / f'misclassifications_{tag}.png'):
        if path.exists():
            mlflow.log_artifact(str(path), artifact_path='evaluation')
    mlflow.log_metrics({f'{tag}_{k}': v for k, v in metrics.items()})
    print(f'--- {tag} ---\n{report}')
    return metrics, matrix

In [ ]:
# 8. Etapa 1 — baseline: EfficientNetB0 ImageNet congelada, solo se entrena la cabeza
inputs = keras.Input(shape=(*IMAGE_SIZE, 3), name='image')
augmentation = keras.Sequential(
    [layers.RandomFlip('horizontal'), layers.RandomRotation(0.05), layers.RandomZoom(0.10)],
    name='train_augmentation',
)
base_model = applications.EfficientNetB0(
    include_top=False, weights='imagenet', input_shape=(*IMAGE_SIZE, 3)
)  # Keras 3: EfficientNet ya incluye el preprocesamiento y espera float32 en [0, 255]
base_model.trainable = False

x = augmentation(inputs)
x = base_model(x, training=False)  # BatchNorm en inferencia incluso durante fit
x = layers.GlobalAveragePooling2D(name='global_average_pooling')(x)
x = layers.Dropout(0.2, name='classification_dropout')(x)
outputs = layers.Dense(len(class_names), activation='softmax', name='classifier', dtype='float32')(x)
model = keras.Model(inputs, outputs, name='efficientnetb0_baseline')

model.compile(optimizer=keras.optimizers.Adam(LEARNING_RATE),
              loss=keras.losses.SparseCategoricalCrossentropy(),
              metrics=[keras.metrics.SparseCategoricalAccuracy(name='accuracy')])

tracker = EmissionsTracker(project_name='leafthealthvision-colab-baseline', output_dir='.', gpu_ids=[])
tracker.start()
t0 = time.perf_counter()
with mlflow.start_run(run_name='colab_baseline') as run_baseline:
    mlflow.log_params({
        'architecture': 'EfficientNetB0', 'weights': 'imagenet', 'include_preprocessing': True,
        'extractor_frozen': True, 'resolution': f'{IMAGE_SIZE[0]}x{IMAGE_SIZE[1]}', 'channels': 'RGB',
        'batch_size': BATCH_SIZE, 'max_epochs': EPOCHS_BASELINE, 'learning_rate': LEARNING_RATE,
        'optimizer': 'Adam', 'loss': 'SparseCategoricalCrossentropy', 'seed': SEED,
        'num_classes': len(class_names), 'augmentation': 'horizontal_flip,rotation_0.05,zoom_0.10',
        'split': 'manifiestos_repo' if MANIFESTS_FROM_REPO else 'estratificado_sin_hoja',
    })
    history = model.fit(
        train_ds, validation_data=val_ds, epochs=EPOCHS_BASELINE, class_weight=class_weight,
        callbacks=[MlflowEpochLogger(),
                   keras.callbacks.EarlyStopping(monitor='val_loss', patience=EARLY_STOPPING_PATIENCE,
                                                 restore_best_weights=True),
                   keras.callbacks.ModelCheckpoint(ARTIFACTS / 'baseline_best.keras',
                                                   monitor='val_loss', save_best_only=True)],
    )
baseline_seconds = time.perf_counter() - t0
try:
    baseline_co2 = tracker.stop() or 0.0
except Exception:  # noqa: BLE001
    baseline_co2 = 0.0
model.save(ARTIFACTS / 'baseline_model.keras')
mlflow.log_metrics({'tiempo_entrenamiento_segundos': round(baseline_seconds, 2),
                    'emisiones_co2_kg': float(baseline_co2)})
baseline_metrics, baseline_matrix = evaluate_stage(model, 'baseline')

In [ ]:
# 9. Etapa 2 — fine-tuning: descongelar las últimas N capas, BatchNorm congelada, lr bajo
base_model.trainable = True
for layer in base_model.layers[:-FT_UNFREEZE_LAST_N]:
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
model.compile(optimizer=keras.optimizers.Adam(FT_LEARNING_RATE),
              loss=keras.losses.SparseCategoricalCrossentropy(),
              metrics=[keras.metrics.SparseCategoricalAccuracy(name='accuracy')])
print('Capas descongeladas del extractor:', len([l for l in base_model.layers if l.trainable]))

tracker = EmissionsTracker(project_name='leafthealthvision-colab-finetune', output_dir='.', gpu_ids=[])
tracker.start()
t0 = time.perf_counter()
with mlflow.start_run(run_name='colab_finetune') as run_finetune:
    mlflow.log_params({
        'architecture': 'EfficientNetB0', 'weights': 'imagenet', 'include_preprocessing': True,
        'extractor_frozen': False, 'unfreeze_last_n': FT_UNFREEZE_LAST_N, 'batchnorm_frozen': True,
        'learning_rate_finetune': FT_LEARNING_RATE, 'resolution': f'{IMAGE_SIZE[0]}x{IMAGE_SIZE[1]}',
        'channels': 'RGB', 'batch_size': BATCH_SIZE, 'max_epochs': EPOCHS_FINETUNE,
        'optimizer': 'Adam', 'loss': 'SparseCategoricalCrossentropy', 'seed': SEED,
        'num_classes': len(class_names), 'augmentation': 'horizontal_flip,rotation_0.05,zoom_0.10',
        'split': 'manifiestos_repo' if MANIFESTS_FROM_REPO else 'estratificado_sin_hoja',
    })
    ft_history = model.fit(
        train_ds, validation_data=val_ds, epochs=EPOCHS_FINETUNE, class_weight=class_weight,
        callbacks=[MlflowEpochLogger(),
                   keras.callbacks.EarlyStopping(monitor='val_loss', patience=FT_EARLY_STOPPING_PATIENCE,
                                                 restore_best_weights=True),
                   keras.callbacks.ModelCheckpoint(ARTIFACTS / 'finetuned_best.keras',
                                                   monitor='val_loss', save_best_only=True)],
    )
ft_seconds = time.perf_counter() - t0
try:
    ft_co2 = tracker.stop() or 0.0
except Exception:  # noqa: BLE001
    ft_co2 = 0.0
model.save(ARTIFACTS / 'finetuned_model.keras')
mlflow.log_metrics({'tiempo_entrenamiento_segundos': round(ft_seconds, 2),
                    'emisiones_co2_kg': float(ft_co2)})
finetune_metrics, finetune_matrix = evaluate_stage(model, 'finetune')

In [ ]:
# 10. Comparación de etapas y pares de confusión más frecuentes
summary = pd.DataFrame([baseline_metrics, finetune_metrics], index=['baseline_congelado', 'fine_tuning'])
display(summary)


def top_confusions(matrix, k=10):
    m = matrix.copy().astype(int)
    np.fill_diagonal(m, 0)
    pairs = []
    for _ in range(k):
        i, j = np.unravel_index(m.argmax(), m.shape)
        if m[i, j] == 0:
            break
        pairs.append((class_names[i], class_names[j], int(m[i, j])))
        m[i, j] = 0
    return pairs


print('Errores más frecuentes (real → predicha):')
for real, pred, n in top_confusions(finetune_matrix):
    print(f'  {n:4d}  {real}  →  {pred}')

## Notas y limitaciones

- Las métricas solo son comparables con el pipeline local y con la Persona A si se usaron los
  **manifiestos del repo** (`data/splits/*.csv`); el fallback estratificado se registra en el parámetro
  `split` de MLflow y sufre fuga por hoja.
- Hipótesis habituales para el análisis de errores: clases con síntomas muy parecidos
  (p. ej. healthy vs. virus de hoja rizada), fondo de la variante `color` menos limpio que `segmented`,
  y desbalance residual mitigado con `class_weight`.
- Réplica local (GPU/CPU propia): `uv run python -m leafthealthvision.training.efficientnetb0` y
  `uv run python -m leafthealthvision.training.finetune` con sus configs; ver README.md.